# Lesson 9: Intro to Neural Networks and Computer Vision

## Recognizing Handwritten Digits with a Simple Neural Network

In this notebook, you will train a simple neural-network-style model to recognize handwritten digits from 0 to 9.

We will use:

- Python
- pandas
- numpy
- matplotlib
- scikit-learn
- Google Colab or Jupyter Notebook

This notebook is designed for high school students who already know Python and basic data science.

---

**Solution Version:** All `# TODO` cells have been completed with working code. The Teacher Solution sections are also kept for reference. This notebook can be executed top-to-bottom without modification.

# Learning Objectives

By the end of this notebook, you will be able to:

1. Explain how a computer represents an image as numbers.
2. Load and explore a small image dataset.
3. Visualize handwritten digit images.
4. Prepare image data for machine learning.
5. Train a simple neural network using `MLPClassifier`.
6. Evaluate the model using accuracy and a confusion matrix.
7. Analyze wrong predictions.
8. Improve the model by changing the number of hidden neurons.
9. Explain how this connects to computer vision projects for STEM competitions.

# Part 1: Background

## What is Computer Vision?

Computer vision is a field of artificial intelligence that helps computers understand images and videos.

Examples:

- Recognizing handwritten digits
- Detecting animals in camera-trap images
- Identifying plant diseases from leaf images
- Detecting road signs for self-driving cars
- Classifying medical images

## How does a computer see an image?

Humans see a handwritten digit as a picture.

A computer sees the same image as a table of numbers.

For a grayscale image:

- Small number = darker pixel
- Large number = brighter pixel

In this lesson, each image is 8 × 8 pixels.

That means each image has:

8 × 8 = 64 numbers

These 64 numbers become the input features for the machine learning model.

## What is a Neural Network?

A neural network is a machine learning model made of connected layers.

A simple way to think about it:

- Input layer: receives the pixel values
- Hidden layer: learns patterns
- Output layer: predicts the digit class

In this notebook, we will use `MLPClassifier`.

MLP means **Multi-Layer Perceptron**, which is a simple type of neural network.

# Part 2: Load the Dataset

We will use the built-in `digits` dataset from scikit-learn.

This dataset contains small images of handwritten digits.

Dataset facts:

- 1,797 images
- Each image is 8 × 8 pixels
- Each image is represented as 64 numbers
- Labels are digits from 0 to 9

In [ ]:
# Import the libraries we need.

# NumPy helps us work with arrays and numbers.
import numpy as np

# Pandas helps us display data like a spreadsheet.
import pandas as pd

# Matplotlib helps us draw images and charts.
import matplotlib.pyplot as plt

# load_digits gives us a small handwritten digit dataset.
from sklearn.datasets import load_digits

# train_test_split helps us divide data into training and test sets.
from sklearn.model_selection import train_test_split

# StandardScaler helps us scale features so the neural network trains better.
from sklearn.preprocessing import StandardScaler

# MLPClassifier is a simple neural network classifier in scikit-learn.
from sklearn.neural_network import MLPClassifier

# These tools help us evaluate the model.
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report

# This makes charts display inside the notebook.
%matplotlib inline

In [ ]:
# Load the digits dataset from scikit-learn.
digits = load_digits()

# TODO 1:
# Store the image feature data in X.
X = digits.data

# TODO 2:
# Store the labels in y.
y = digits.target

# TODO 3:
# Print the shapes of X and y.
print("X shape:", X.shape)
print("y shape:", y.shape)

# TODO 4:
# Print the target names.
print("Target names:", digits.target_names)

# Part 3: Explore the Data

Before training a model, we should understand the data.

We will inspect:

1. The shape of the image data
2. The labels
3. The pixel values
4. Some sample digit images

In [ ]:
# TODO 5:
# Create a pandas DataFrame from X.
# Each row is one image. Each column is one pixel feature.
digits_df = pd.DataFrame(X)

# TODO 6:
# Display the first 5 rows of the DataFrame.
digits_df.head()

In [ ]:
# TODO 7:
# Print the label for the first image.
print("Label of the first image:", y[0])

# TODO 8:
# Print the 64 pixel values for the first image.
print("Pixel values of the first image:")
print(X[0])

In [ ]:
# TODO 9:
# Display the first image.
# Hint: digits.images[0] is already in 8 x 8 image form.
plt.imshow(digits.images[0], cmap="gray")
plt.title("Label: " + str(y[0]))
plt.axis("off")
plt.show()

In [ ]:
# TODO 10:
# Display the first 10 images in a grid.

plt.figure(figsize=(10, 4))

for i in range(10):
    plt.subplot(2, 5, i + 1)

    # show image i
    plt.imshow(digits.images[i], cmap="gray")

    # show label i
    plt.title("Label: " + str(y[i]))

    plt.axis("off")

plt.tight_layout()
plt.show()

## Quick Check

Answer these questions before moving on:

1. How many images are in the dataset?
2. How many features does each image have?
3. Why does each image have 64 features?
4. What does the label represent?

# Part 4: Prepare the Data

Machine learning models need training data and test data.

- Training data: used to teach the model
- Test data: used to check if the model can make good predictions on new examples

We will use:

- 80% training data
- 20% test data

We will also scale the pixel values because neural networks usually train better when features are scaled.

In [ ]:
# TODO 11:
# Split the data into training and test sets.
# Use test_size=0.2, random_state=42, and stratify=y.

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# TODO 12:
# Print the shapes of the training and test sets.
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

In [ ]:
# Create a StandardScaler object.
scaler = StandardScaler()

# TODO 13:
# Fit the scaler on the training data and transform the training data.
X_train_scaled = scaler.fit_transform(X_train)

# TODO 14:
# Transform the test data using the same scaler.
X_test_scaled = scaler.transform(X_test)

# Print the mean and standard deviation of the scaled training data.
# The mean should be close to 0, and the standard deviation should be close to 1.
print("Mean of scaled training data:", np.round(X_train_scaled.mean(), 4))
print("Standard deviation of scaled training data:", np.round(X_train_scaled.std(), 4))

# Part 5: Train the Model

Now we will train a simple neural network.

We will use `MLPClassifier`.

Important parameter:

```python
hidden_layer_sizes=(32,)
```

This means the model has one hidden layer with 32 neurons.

The model will learn patterns from the 64 pixel values and try to predict the correct digit.

In [ ]:
# TODO 15:
# Create an MLPClassifier model.
# Use hidden_layer_sizes=(32,), max_iter=500, random_state=42.

model = MLPClassifier(
    hidden_layer_sizes=(32,),
    max_iter=500,
    random_state=42
)

# TODO 16:
# Train the model on the scaled training data.
model.fit(X_train_scaled, y_train)

print("Model training complete!")

# Part 6: Evaluate the Model

After training, we need to test the model.

We will calculate:

1. Accuracy
2. Confusion matrix
3. Classification report

Accuracy tells us the percentage of predictions that are correct.

A confusion matrix tells us which digits the model confused with other digits.

In [ ]:
# TODO 17:
# Use the model to predict labels for the scaled test data.
y_pred = model.predict(X_test_scaled)

# TODO 18:
# Calculate accuracy.
accuracy = accuracy_score(y_test, y_pred)

print("Test Accuracy:", accuracy)

In [ ]:
# TODO 19:
# Create a confusion matrix.
cm = confusion_matrix(y_test, y_pred)

# Display the confusion matrix.
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=digits.target_names
)

disp.plot()
plt.title("Confusion Matrix for Digit Classification")
plt.show()

In [ ]:
# Print a classification report.
# Precision, recall, and F1-score help us evaluate each digit class.

print(classification_report(y_test, y_pred))

In [ ]:
# TODO 20:
# Find the indexes where the model made wrong predictions.
wrong_indexes = np.where(y_test != y_pred)[0]

print("Number of wrong predictions:", len(wrong_indexes))

# Display up to 5 wrong predictions.
plt.figure(figsize=(10, 3))

for i, wrong_index in enumerate(wrong_indexes[:5]):
    plt.subplot(1, 5, i + 1)

    # X_test[wrong_index] is flattened, so we reshape it back into 8 x 8.
    plt.imshow(X_test[wrong_index].reshape(8, 8), cmap="gray")

    plt.title("True: " + str(y_test[wrong_index]) + "\nPred: " + str(y_pred[wrong_index]))
    plt.axis("off")

plt.tight_layout()
plt.show()

# Part 7: Improve the Model

A machine learning model has settings called **hyperparameters**.

For our neural network, one important hyperparameter is the number of hidden neurons.

We will test different hidden layer sizes:

- 16 neurons
- 32 neurons
- 64 neurons
- 128 neurons

Then we will compare the accuracy.

In [ ]:
# List of hidden layer sizes to test.
hidden_sizes = [16, 32, 64, 128]

# This list will store the results.
results = []

for size in hidden_sizes:
    # Create a neural network with one hidden layer.
    test_model = MLPClassifier(
        hidden_layer_sizes=(size,),
        max_iter=500,
        random_state=42
    )

    # Train the model.
    test_model.fit(X_train_scaled, y_train)

    # Make predictions.
    test_pred = test_model.predict(X_test_scaled)

    # Calculate accuracy.
    test_accuracy = accuracy_score(y_test, test_pred)

    # Save the result.
    results.append({
        "Hidden Neurons": size,
        "Accuracy": test_accuracy
    })

# Convert results to a DataFrame.
results_df = pd.DataFrame(results)

# Display the results.
results_df

In [ ]:
# Plot the accuracy results.

plt.figure(figsize=(8, 5))
plt.plot(results_df["Hidden Neurons"], results_df["Accuracy"], marker="o")
plt.xlabel("Number of Hidden Neurons")
plt.ylabel("Test Accuracy")
plt.title("Model Accuracy vs Number of Hidden Neurons")
plt.grid(True)
plt.show()

## Think Like a Researcher

Answer these questions:

1. Which hidden layer size gave the best accuracy?
2. Did more neurons always improve the model?
3. What might happen if the model becomes too large?
4. Why should we test several model settings in a competition project?

# Part 8: Reflection Questions

Write your answers in a markdown cell below.

## Concept Questions

1. What is computer vision?
2. How does a computer represent a grayscale image?
3. Why does each digit image in this dataset have 64 features?
4. What is the purpose of the training set?
5. What is the purpose of the test set?

## Neural Network Questions

6. What does `MLPClassifier` do?
7. What does `hidden_layer_sizes=(32,)` mean?
8. Why do we scale data before training a neural network?
9. What does the confusion matrix show?
10. Which digits were hardest for the model to classify?

## STEM Competition Questions

11. What real-world problem could image classification help solve?
12. What are the limitations of this small digits dataset?
13. What kind of dataset would you need for a stronger STEM competition project?
14. Why is error analysis important?
15. What ethical concerns can appear in computer vision projects?

In [ ]:
# Student Reflection Cell
# Write your answers below as comments or use a markdown cell.

# 1.
# 2.
# 3.
# 4.
# 5.
# 6.
# 7.
# 8.
# 9.
# 10.
# 11.
# 12.
# 13.
# 14.
# 15.

# Part 9: Challenge Task

## Challenge: Compare Neural Network with Random Forest

In the textbook, image classification with scikit-learn is introduced using a Random Forest classifier.

Your challenge is to compare:

1. Neural Network: `MLPClassifier`
2. Random Forest: `RandomForestClassifier`

Questions:

1. Which model has higher accuracy?
2. Which model is easier to train?
3. Which model would you use for a small beginner project?
4. Why might neural networks become more powerful for larger image datasets?

In [ ]:
# Challenge Task Starter Code

from sklearn.ensemble import RandomForestClassifier

# TODO 21:
# Create a RandomForestClassifier.
# Use n_estimators=100 and random_state=42.
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

# TODO 22:
# Train the random forest model.
# Random Forest does not require scaling, so use X_train instead of X_train_scaled.
rf_model.fit(X_train, y_train)

# TODO 23:
# Make predictions on X_test.
rf_pred = rf_model.predict(X_test)

# TODO 24:
# Calculate Random Forest accuracy.
rf_accuracy = accuracy_score(y_test, rf_pred)

print("Neural Network Accuracy:", accuracy)
print("Random Forest Accuracy:", rf_accuracy)

# Teacher Solution Section

The following cells contain complete solutions.

Students should try the TODO sections first before looking at the solutions.

## Teacher Solution: Complete Notebook Code

In [ ]:
# ===============================
# Teacher Solution: Complete Code
# ===============================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report
from sklearn.ensemble import RandomForestClassifier

%matplotlib inline

# Load the dataset.
digits = load_digits()

# Store features and labels.
X = digits.data
y = digits.target

# Print basic dataset information.
print("X shape:", X.shape)
print("y shape:", y.shape)
print("Target names:", digits.target_names)

# Create a DataFrame to inspect the first few rows.
digits_df = pd.DataFrame(X)
display(digits_df.head())

# Display the first image.
plt.imshow(digits.images[0], cmap="gray")
plt.title("Label: " + str(y[0]))
plt.axis("off")
plt.show()

# Display the first 10 images.
plt.figure(figsize=(10, 4))

for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(digits.images[i], cmap="gray")
    plt.title("Label: " + str(y[i]))
    plt.axis("off")

plt.tight_layout()
plt.show()

# Split data into training and test sets.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

# Scale the data.
scaler = StandardScaler()

# Fit only on the training data.
X_train_scaled = scaler.fit_transform(X_train)

# Transform test data using the same scaler.
X_test_scaled = scaler.transform(X_test)

print("Mean of scaled training data:", np.round(X_train_scaled.mean(), 4))
print("Standard deviation of scaled training data:", np.round(X_train_scaled.std(), 4))

# Create and train the neural network.
model = MLPClassifier(
    hidden_layer_sizes=(32,),
    max_iter=500,
    random_state=42
)

model.fit(X_train_scaled, y_train)

print("Model training complete!")

# Make predictions.
y_pred = model.predict(X_test_scaled)

# Calculate accuracy.
accuracy = accuracy_score(y_test, y_pred)

print("Test Accuracy:", accuracy)

# Create confusion matrix.
cm = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=digits.target_names
)

disp.plot()
plt.title("Confusion Matrix for Digit Classification")
plt.show()

# Print classification report.
print(classification_report(y_test, y_pred))

# Find wrong predictions.
wrong_indexes = np.where(y_test != y_pred)[0]

print("Number of wrong predictions:", len(wrong_indexes))

# Display up to 5 wrong predictions.
plt.figure(figsize=(10, 3))

for i, wrong_index in enumerate(wrong_indexes[:5]):
    plt.subplot(1, 5, i + 1)
    plt.imshow(X_test[wrong_index].reshape(8, 8), cmap="gray")
    plt.title("True: " + str(y_test[wrong_index]) + "\nPred: " + str(y_pred[wrong_index]))
    plt.axis("off")

plt.tight_layout()
plt.show()

## Teacher Solution: Model Improvement

In [ ]:
# Test several hidden layer sizes.

hidden_sizes = [16, 32, 64, 128]
results = []

for size in hidden_sizes:
    test_model = MLPClassifier(
        hidden_layer_sizes=(size,),
        max_iter=500,
        random_state=42
    )

    test_model.fit(X_train_scaled, y_train)
    test_pred = test_model.predict(X_test_scaled)
    test_accuracy = accuracy_score(y_test, test_pred)

    results.append({
        "Hidden Neurons": size,
        "Accuracy": test_accuracy
    })

results_df = pd.DataFrame(results)
display(results_df)

plt.figure(figsize=(8, 5))
plt.plot(results_df["Hidden Neurons"], results_df["Accuracy"], marker="o")
plt.xlabel("Number of Hidden Neurons")
plt.ylabel("Test Accuracy")
plt.title("Model Accuracy vs Number of Hidden Neurons")
plt.grid(True)
plt.show()

## Teacher Solution: Challenge Task

In [ ]:
# Compare MLPClassifier with RandomForestClassifier.

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

# Random Forest does not require scaling.
rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

rf_accuracy = accuracy_score(y_test, rf_pred)

print("Neural Network Accuracy:", accuracy)
print("Random Forest Accuracy:", rf_accuracy)

# Show Random Forest confusion matrix.
rf_cm = confusion_matrix(y_test, rf_pred)

rf_disp = ConfusionMatrixDisplay(
    confusion_matrix=rf_cm,
    display_labels=digits.target_names
)

rf_disp.plot()
plt.title("Random Forest Confusion Matrix")
plt.show()

# Teacher Notes

Expected results:

- Neural network accuracy is usually around 0.96 to 0.98.
- Random forest accuracy is also usually strong on this small dataset.
- Results may vary slightly depending on random state and environment.

Teaching emphasis:

1. An image can be represented as numbers.
2. Computer vision is classification when the goal is to assign a label to an image.
3. Neural networks can learn image patterns, but simple scikit-learn models can also work well on small image datasets.
4. A strong STEM competition project should include error analysis, limitations, and real-world impact, not only model accuracy.

Possible discussion:

- Why are 8 × 8 images much simpler than real photos?
- What changes when we use color images?
- Why do real computer vision systems often use convolutional neural networks?
- How could students turn image classification into a science fair project?